In [9]:
"""
TF-IDF features + logistic regression. 

Usage:
    python3 train_baseline_classifier.py juliet_hallucination_dataset_1k.json
"""
import sys
import os
import json
import random
import pickle
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

RANDOM_SEED = 42
TEST_FRACTION = 0.15
MODEL_SAVE_PATH = "baseline_classifier.pkl"
DEFAULT_DATASET_PATH = "juliet_hallucination_dataset_1k.json"
EXCLUDED_TYPES = {"DUPLICATE_OR_DEGENERATE", "EMPTY_EXTRACTION", "GENERATION_ERROR", "AST_PARSE_ERROR"}


def load_and_prepare_rows(path):
    with open(path, "r", encoding="utf-8") as f:
        raw_rows = json.load(f)

    kept, excluded_counts = [], {}
    for row in raw_rows:
        h_type = row.get("hallucination_type", "NONE")
        if h_type in EXCLUDED_TYPES:
            excluded_counts[h_type] = excluded_counts.get(h_type, 0) + 1
            continue

        extracted = (row.get("extracted_code", "") or "").strip()
        if not extracted:
            excluded_counts["_blank_extracted_code"] = excluded_counts.get("_blank_extracted_code", 0) + 1
            continue

        text = f"{row.get('prompt_context', '')}\n{extracted}"
        case_id = row.get("case_identifier", "unknown")
        cwe_group = case_id.split("__")[0]

        kept.append({
            "text": text,
            "label": int(bool(row["is_hallucinated"])),
            "cwe_group": cwe_group,
            "case_identifier": case_id,
        })

    print(f"[+] Loaded {len(raw_rows)} raw rows.")
    if excluded_counts:
        print("[+] Excluded (pipeline-failure signals, not hallucination judgments):")
        for t, c in sorted(excluded_counts.items(), key=lambda x: -x[1]):
            print(f"    {t}: {c}")
    print(f"[+] {len(kept)} rows retained.")
    print(f"[+] Label balance: {dict(Counter(r['label'] for r in kept))}")
    return kept


def group_aware_split(rows, test_fraction=TEST_FRACTION, seed=RANDOM_SEED):
    """Same group-aware split as the transformer script: hold out whole CWE
    families rather than random rows, so near-duplicate flow-variants of
    the same template can't leak across train/test."""
    groups = sorted(set(r["cwe_group"] for r in rows))
    rng = random.Random(seed)
    rng.shuffle(groups)

    target_test_size = int(len(rows) * test_fraction)
    test_groups, running_count = set(), 0
    for g in groups:
        if running_count >= target_test_size:
            break
        group_rows = [r for r in rows if r["cwe_group"] == g]
        test_groups.add(g)
        running_count += len(group_rows)

    train_rows = [r for r in rows if r["cwe_group"] not in test_groups]
    test_rows = [r for r in rows if r["cwe_group"] in test_groups]
    print(f"[+] Group-aware split: {len(groups)} CWE groups, {len(test_groups)} held out.")
    print(f"[+] Train rows: {len(train_rows)}  |  Eval rows: {len(test_rows)}")
    return train_rows, test_rows


def main():
    if len(sys.argv) == 2 and os.path.exists(sys.argv[1]):
        dataset_path = sys.argv[1]
    else:
        dataset_path = DEFAULT_DATASET_PATH
        print(f"[*] No valid CLI file argument found -- using DEFAULT_DATASET_PATH: {dataset_path}")

    if not os.path.exists(dataset_path):
        print(f"[-] {dataset_path} not found. Either place your dataset file there, "
              f"or edit DEFAULT_DATASET_PATH near the top of this script.")
        sys.exit(1)

    rows = load_and_prepare_rows(dataset_path)
    if len(rows) < 10:
        print("[!] Very few rows after filtering -- results will be unstable regardless of model choice.")

    train_rows, eval_rows = group_aware_split(rows)

    train_texts = [r["text"] for r in train_rows]
    train_labels = [r["label"] for r in train_rows]
    eval_texts = [r["text"] for r in eval_rows]
    eval_labels = [r["label"] for r in eval_rows]

    vectorizer = TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=1,
    )
    X_train = vectorizer.fit_transform(train_texts)
    X_eval = vectorizer.transform(eval_texts)


    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )
    clf.fit(X_train, train_labels)

    train_preds = clf.predict(X_train)
    eval_preds = clf.predict(X_eval)

    print("\n[+] TRAIN split (sanity check -- should look reasonable, not necessarily perfect):")
    print(f"    accuracy:  {accuracy_score(train_labels, train_preds):.3f}")
    print(f"    f1:        {f1_score(train_labels, train_preds, zero_division=0):.3f}")

    print("\n[+] EVAL split (held-out CWE groups):")
    print(f"    accuracy:  {accuracy_score(eval_labels, eval_preds):.3f}")
    print(f"    f1:        {f1_score(eval_labels, eval_preds, zero_division=0):.3f}")
    print(f"    precision: {precision_score(eval_labels, eval_preds, zero_division=0):.3f}")
    print(f"    recall:    {recall_score(eval_labels, eval_preds, zero_division=0):.3f}")
    print(f"    confusion matrix (rows=true, cols=pred, [0,1] order):")
    print(f"    {confusion_matrix(eval_labels, eval_preds, labels=[0, 1])}")
    print()
    print(classification_report(eval_labels, eval_preds, target_names=["not_hallucinated", "hallucinated"], zero_division=0))

    feature_names = vectorizer.get_feature_names_out()
    coefs = clf.coef_[0]
    top_pos_idx = coefs.argsort()[-15:][::-1]
    top_neg_idx = coefs.argsort()[:15]
    print("\n[+] Top features pushing toward HALLUCINATED:")
    for i in top_pos_idx:
        print(f"    {feature_names[i]:30s}  weight={coefs[i]:+.3f}")
    print("\n[+] Top features pushing toward NOT hallucinated:")
    for i in top_neg_idx:
        print(f"    {feature_names[i]:30s}  weight={coefs[i]:+.3f}")

    with open(MODEL_SAVE_PATH, "wb") as f:
        pickle.dump({"vectorizer": vectorizer, "classifier": clf}, f)
    print(f"\n[+] Saved vectorizer + classifier to {MODEL_SAVE_PATH}")


if __name__ == "__main__":
    main()

[*] No valid CLI file argument found -- using DEFAULT_DATASET_PATH: juliet_hallucination_dataset_1k.json
[+] Loaded 1000 raw rows.
[+] Excluded (pipeline-failure signals, not hallucination judgments):
    EMPTY_EXTRACTION: 51
    DUPLICATE_OR_DEGENERATE: 22
[+] 927 rows retained.
[+] Label balance: {0: 486, 1: 441}
[+] Group-aware split: 46 CWE groups, 6 held out.
[+] Train rows: 785  |  Eval rows: 142

[+] TRAIN split (sanity check -- should look reasonable, not necessarily perfect):
    accuracy:  0.762
    f1:        0.747

[+] EVAL split (held-out CWE groups):
    accuracy:  0.648
    f1:        0.590
    precision: 0.643
    recall:    0.545
    confusion matrix (rows=true, cols=pred, [0,1] order):
    [[56 20]
 [30 36]]

                  precision    recall  f1-score   support

not_hallucinated       0.65      0.74      0.69        76
    hallucinated       0.64      0.55      0.59        66

        accuracy                           0.65       142
       macro avg       0.65  

# Roberta

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import re
import json
import random
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATASET_PATH = "juliet_hallucination_dataset_1k.json"
GOLD_SET_PATH = "gold_set.json"
MODEL_NAME = "roberta-base"
OUTPUT_DIR = "./lora_hallucination_detector_roberta"
FINAL_SAVE_PATH = "./final_lora_detector_weights_roberta"
MAX_LENGTH = 512
TEST_FRACTION = 0.15
RANDOM_SEED = 42

# Same pipeline-failure exclusion list as before: these are generation/
# extraction failures, not a judgment about the code itself.
EXCLUDED_TYPES = {"DUPLICATE_OR_DEGENERATE", "EMPTY_EXTRACTION", "GENERATION_ERROR", "AST_PARSE_ERROR"}

random.seed(RANDOM_SEED)


# ==========================================
# 2. LOAD + FILTER + BUILD INPUT TEXT
# ==========================================
def load_and_prepare_rows(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {path}. Run mine_hallucinations.py first!")

    with open(path, "r", encoding="utf-8") as f:
        raw_rows = json.load(f)

    kept, excluded_counts = [], {}
    for row in raw_rows:
        h_type = row.get("hallucination_type", "NONE")
        if h_type in EXCLUDED_TYPES:
            excluded_counts[h_type] = excluded_counts.get(h_type, 0) + 1
            continue

        extracted = row.get("extracted_code", "") or ""
        if not extracted.strip():
            excluded_counts["_blank_extracted_code_fallthrough"] = (
                excluded_counts.get("_blank_extracted_code_fallthrough", 0) + 1
            )
            continue

        text = (
            f"### Vulnerable code:\n{row.get('prompt_context', '')}\n\n"
            f"### Model's remediation:\n{extracted}"
        )

        case_id = row.get("case_identifier", "unknown")
        cwe_group = case_id.split("__")[0]

        kept.append({
            "text": text,
            "labels": int(bool(row["is_hallucinated"])),
            "hallucination_type": h_type,
            "cwe_group": cwe_group,
            "case_identifier": case_id,
        })

    print(f"[+] Loaded {len(raw_rows)} raw rows.")
    if excluded_counts:
        print("[+] Excluded from training (pipeline-failure signals, not hallucination judgments):")
        for t, c in sorted(excluded_counts.items(), key=lambda x: -x[1]):
            print(f"    {t}: {c}")
    print(f"[+] {len(kept)} rows retained for training.")

    label_counts = {}
    for r in kept:
        label_counts[r["labels"]] = label_counts.get(r["labels"], 0) + 1
    print(f"[+] Retained label balance: {label_counts}")

    return kept


# ==========================================
# 3. GROUP-AWARE TRAIN/TEST SPLIT
# ==========================================
def group_aware_split(rows, test_fraction=TEST_FRACTION, seed=RANDOM_SEED):
    groups = sorted(set(r["cwe_group"] for r in rows))
    rng = random.Random(seed)
    rng.shuffle(groups)

    target_test_size = int(len(rows) * test_fraction)
    test_groups, running_count = set(), 0
    for g in groups:
        if running_count >= target_test_size:
            break
        group_rows = [r for r in rows if r["cwe_group"] == g]
        test_groups.add(g)
        running_count += len(group_rows)

    train_rows = [r for r in rows if r["cwe_group"] not in test_groups]
    test_rows = [r for r in rows if r["cwe_group"] in test_groups]

    print(f"[+] Group-aware split: {len(groups)} distinct CWE groups total, "
          f"{len(test_groups)} held out for eval.")
    print(f"[+] Train rows: {len(train_rows)}  |  Eval rows: {len(test_rows)}")
    return train_rows, test_rows


# ==========================================
# 4. MAIN TRAINING PIPELINE
# ==========================================
def main():
    rows = load_and_prepare_rows(DATASET_PATH)
    train_rows, eval_rows = group_aware_split(rows)

    train_dataset = Dataset.from_list(train_rows)
    eval_dataset = Dataset.from_list(eval_rows)

    print(f"[+] Loading tokenizer/model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

    tokenized_train = train_dataset.map(tokenize_fn, batched=True)
    tokenized_eval = eval_dataset.map(tokenize_fn, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["query", "value"],
        modules_to_save=["classifier"],
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    
    train_labels = np.array(tokenized_train["labels"])
    n_pos = int((train_labels == 1).sum())
    n_neg = int((train_labels == 0).sum())
    total = n_pos + n_neg
    if n_pos > 0 and n_neg > 0:
        raw_weights = [total / (2.0 * n_neg), total / (2.0 * n_pos)]
        max_ratio = 5.0
        min_w, max_w = min(raw_weights), max(raw_weights)
        if min_w > 0 and max_w / min_w > max_ratio:
            scale = max_ratio / (max_w / min_w)
            raw_weights = [w * scale if w == max_w else w for w in raw_weights]
        class_weights = torch.tensor(raw_weights, dtype=torch.float32)
    else:
        class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32)
    print(f"[+] Fixed class weights (neg, pos): {class_weights.tolist()}")
    print(f"[+] Train label counts -- neg: {n_neg}, pos: {n_pos}, total: {total}")

    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    mcc_metric = evaluate.load("matthews_correlation")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
            "f1": f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"],
            "precision": precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"],
            "recall": recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"],
            "matthews_correlation": mcc_metric.compute(predictions=predictions, references=labels)["matthews_correlation"],
        }

    class FixedWeightTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")
            weights = self.class_weights.to(logits.device).to(logits.dtype)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    PER_DEVICE_BATCH_SIZE = 8
    NUM_EPOCHS = 8

    steps_per_epoch = max(1, -(-len(tokenized_train) // PER_DEVICE_BATCH_SIZE))  # ceil division
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = max(1, int(0.1 * total_steps))
    print(f"[+] Steps/epoch: {steps_per_epoch}  |  Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=1.5e-4,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_steps=warmup_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="matthews_correlation",
        greater_is_better=True,
        fp16=False,
        bf16=True,
        use_cpu=False,
        logging_steps=10,
        report_to="none",
    )

    trainer = FixedWeightTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    print("\n[*] Training...")
    trainer.train()

    print("\n[*] Eval-split (auto-labeled) results:")
    print(trainer.evaluate())

    model.save_pretrained(FINAL_SAVE_PATH)
    tokenizer.save_pretrained(FINAL_SAVE_PATH)
    print(f"\n[+] Adapter saved to: {FINAL_SAVE_PATH}")

    # ==========================================
    # 5. GOLD-SET EVALUATION (if available)
    # ==========================================
    if os.path.exists(GOLD_SET_PATH):
        print(f"\n[*] Found {GOLD_SET_PATH} -- running gold-set evaluation...")
        with open(GOLD_SET_PATH, "r", encoding="utf-8") as f:
            gold_rows = json.load(f)

        gold_texts, gold_labels = [], []
        for row in gold_rows:
            text = (
                f"### Vulnerable code:\n{row.get('prompt_context', '')}\n\n"
                f"### Model's remediation:\n{row.get('extracted_code', '')}"
            )
            gold_texts.append(text)
            gold_labels.append(int(bool(row["human_label"])))

        gold_dataset = Dataset.from_dict({"text": gold_texts, "labels": gold_labels})
        tokenized_gold = gold_dataset.map(tokenize_fn, batched=True)

        gold_results = trainer.evaluate(eval_dataset=tokenized_gold)
        print("\n[+] GOLD-SET results (measured against human labels, not auto-labels):")
        for k, v in gold_results.items():
            print(f"    {k}: {v}")
    else:
        print(f"\n[!] No {GOLD_SET_PATH} found -- skipping gold-set evaluation.")
        print("    The eval-split metrics above only measure agreement with the")
        print("    auto-labeling pipeline, not real-world correctness.")


if __name__ == "__main__":
    main()

[+] Loaded 1000 raw rows.
[+] Excluded from training (pipeline-failure signals, not hallucination judgments):
    EMPTY_EXTRACTION: 51
    DUPLICATE_OR_DEGENERATE: 22
[+] 927 rows retained for training.
[+] Retained label balance: {0: 486, 1: 441}
[+] Group-aware split: 46 distinct CWE groups total, 6 held out for eval.
[+] Train rows: 785  |  Eval rows: 142
[+] Loading tokenizer/model: roberta-base


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9840.39it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,181,954 || all params: 125,829,124 || trainable%: 0.9393
[+] Fixed class weights (neg, pos): [0.957317054271698, 1.0466666221618652]
[+] Train label counts -- neg: 410, pos: 375, total: 785
[+] Steps/epoch: 99  |  Total steps: 792  |  Warmup steps: 79

[*] Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Matthews Correlation
1,0.723659,0.719950,0.464789,0.634615,0.464789,1.000000,0.000000
2,0.676387,0.699822,0.464789,0.634615,0.464789,1.000000,0.000000
3,0.694359,0.691403,0.535211,0.000000,0.000000,0.000000,0.000000
4,0.681821,0.689425,0.535211,0.175000,0.500000,0.106061,0.023348
5,0.681970,0.685371,0.577465,0.411765,0.583333,0.318182,0.138513
6,0.698373,0.688173,0.521127,0.227273,0.454545,0.151515,-0.008794
7,0.672901,0.682460,0.605634,0.500000,0.608696,0.424242,0.199726
8,0.671532,0.687527,0.556338,0.350515,0.548387,0.257576,0.088578


c:\Users\Jennifer_Nishimura\Documents\DATASCI266\final_project\W266-Final-Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



[*] Eval-split (auto-labeled) results:


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Matthews Correlation
0.671532,0.682460,8,0.605634,0.500000,0.608696,0.424242,0.199726


{'eval_loss': 0.6824600696563721, 'eval_accuracy': 0.6056338028169014, 'eval_f1': 0.5, 'eval_precision': 0.6086956521739131, 'eval_recall': 0.42424242424242425, 'eval_matthews_correlation': 0.1997259784038895}

[+] Adapter saved to: ./final_lora_detector_weights_roberta

[!] No gold_set.json found -- skipping gold-set evaluation.
    The eval-split metrics above only measure agreement with the
    auto-labeling pipeline, not real-world correctness.
